# Fine-tune YOLO26n: 48-class home/garage lost & found (Kaggle free GPU)

Companion to `docs/FINE_TUNING_YOLO_GUIDE.md` in the `vision-assist-v1` repo. Run top to bottom on a Kaggle Notebook with **Accelerator = GPU T4 x2** (or P100) enabled (Settings panel, right sidebar) — GPU access needs phone-number verification on your Kaggle account.

What this notebook does: downloads LVIS + COCO via Ultralytics' own dataset machinery, runs `prepare_dataset.py` to filter/remap/merge them into the project's curated 48-class scheme (20 of COCO's 80 classes kept + 28 new home/garage classes — NOT the original 80+3 scheme; see Guide Section 2 for why most of COCO's 80 classes don't belong in a home lost-and-found use case), trains with `freeze=10`, then runs the three-way eval harness against whatever held-out sets you've uploaded as a Kaggle Dataset (see `finetuning/README.md` for how to prepare and attach those).

Nothing in this repo has executed this yet — treat every cell's expected output in comments as what *should* happen, not a guarantee.

In [ ]:
!pip install -q ultralytics==8.4.106
import ultralytics
ultralytics.checks()

## 1. Force Ultralytics to download + convert LVIS and COCO

`lvis.yaml` is ~20.7GB — this is the slow step. We only need Ultralytics' own download/convert machinery to run once; one epoch is the simplest reliable trigger. Interrupt the cell (Kaggle: stop button) right after it logs `Epoch 1/1` starting, or just let the throwaway epoch finish — either way the dataset is now on disk at whatever `ultralytics` settings.yaml points `datasets_dir` to (print it below).

In [ ]:
from ultralytics import YOLO, settings
print("datasets_dir:", settings["datasets_dir"])

model = YOLO("yolo26n.pt")
model.train(data="lvis.yaml", epochs=1, imgsz=640)  # triggers LVIS download+convert; interrupt after epoch 1 starts

In [ ]:
model = YOLO("yolo26n.pt")
model.train(data="coco.yaml", epochs=1, imgsz=640)  # triggers COCO download+convert (train2017 + val2017); interrupt after epoch 1 starts

## 2. Build the merged 48-class training dataset

Upload `finetuning/prepare_dataset.py` from this repo as a Kaggle Dataset (or paste its contents into a cell) before running this. Adjust `--lvis-root`/`--coco-root` to match whatever path cell 1 printed as `datasets_dir`. This keeps 20 of COCO's 80 classes (person/bicycle/backpack/.../cat/dog — see `KEPT_COCO_CLASS_NAMES` in the script) and adds 28 new classes for things actually searched for at home or in a garage (keys/wallet/glasses/watch/screwdriver/.../ladder — see `NOVEL_CLASSES`), all sourced from real, verified LVIS class indices.

In [ ]:
DATASETS_DIR = settings["datasets_dir"]  # from cell 1's print

!python prepare_dataset.py \
    --lvis-root {DATASETS_DIR}/lvis \
    --coco-root {DATASETS_DIR}/coco \
    --output-dir /kaggle/working/lostfound \
    --coco-sample-size 750

## 2b. Build the forgetting-check set (fully automatic)

Unlike the golden set (which needs real self-captured photos), this one needs no new annotation — it's just a filtered/remapped sample of COCO's own val2017 ground truth. Upload `finetuning/prepare_coco_forgetting_check.py` alongside `prepare_dataset.py`.

In [ ]:
!python prepare_coco_forgetting_check.py \
    --coco-root {DATASETS_DIR}/coco \
    --output-dir /kaggle/working/coco_forgetting_check \
    --sample-size 75

## 3. Train

`freeze=10` — see `docs/FINE_TUNING_YOLO_GUIDE.md` Section 4 for the exact frozen/trainable parameter breakdown (1,115,744 frozen / 1,456,536 trainable, 56.62% of the model trains). A Kaggle session caps at roughly 9-12 hours depending on source; a nano model on a low-thousands-of-images dataset should finish 50-100 epochs on a single T4 well inside that either way (Section 8 estimates 1-3 hours).

In [ ]:
model = YOLO("yolo26n.pt")
results = model.train(
    data="/kaggle/working/lostfound/lostfound.yaml",
    epochs=100,
    imgsz=640,
    freeze=10,
    optimizer="AdamW",
    lr0=0.001,
    patience=15,
    project="/kaggle/working/runs",
    name="lostfound_yolo26n",
)

## 4. Sanity-check the fine-tuned checkpoint

Quick check that the model actually learned the new classes and didn't regress the kept ones — full three-way comparison against the golden set lives in `finetuning/evaluate.py`, run separately (see `finetuning/README.md`).

In [ ]:
best = YOLO("/kaggle/working/runs/lostfound_yolo26n/weights/best.pt")

metrics = best.val(data="/kaggle/working/lostfound/lostfound.yaml", split="val")
print("Training-set val -- mAP50:", metrics.box.map50, "mAP50-95:", metrics.box.map)
print("per-class mAP50-95 for the 28 novel classes (indices 20-47):", metrics.box.maps[20:48])

forgetting_metrics = best.val(data="/kaggle/working/coco_forgetting_check/lostfound.yaml", split="val")
print("COCO forgetting-check mAP50-95 (indices 0-19 only; 20-47 have no instances here):", forgetting_metrics.box.map)

## 5. Download the checkpoint

`best.pt` lands under Kaggle's output tab (`/kaggle/working/runs/lostfound_yolo26n/weights/best.pt`) and downloads from there when the notebook finishes. Next step is Section 9 of `FINE_TUNING_YOLO_GUIDE.md`: host it (Hugging Face Hub recommended) and point `FINE_TUNED_YOLO_MODEL_PATH` at it.